# Khmer AI-Text Detector — Notebook 2: Fine-tune XLM-RoBERTa (step 6)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cheamenglyzz-spec/khmer-ai-text-detector/blob/step2-ai-generation/notebooks/02_finetune_xlmr.ipynb)

**Goal:** fine-tune `xlm-roberta-base` to tell human-written Khmer text (label 0) from AI-generated text (label 1), then score it on val, test and the held-out generator (`test_ood`) with the same metrics as the step-5 baseline.

This notebook is only a driver. All the logic is in `src/finetune.py`, which it gets in one of two ways (`CODE_SOURCE` in the settings cell):
- `"drive"` (default): unzips `colab_bundle.zip` from Google Drive. The bundle holds the code and the four split files; make it locally with `python src/make_colab_bundle.py` and drag it into the root of **My Drive**.
- `"github"`: clones the repo (public, or private with a `GITHUB_TOKEN` secret) and reads the splits from `MyDrive/khmer-ai-text-detector/splits/`.

**Needs**
- A **GPU runtime**: `Runtime → Change runtime type → T4 GPU`.
- The data: with `CODE_SOURCE = "drive"` it is inside the bundle. With `"github"`, copy `train.jsonl`, `val.jsonl`, `test.jsonl`, `test_ood.jsonl` (gitignored, made by `src/split.py`) to `MyDrive/khmer-ai-text-detector/splits/`; if they are missing you can upload them instead.

**Time:** roughly 5–10 minutes on a T4 for the current small splits (about 1,400 training texts), and 30–45 minutes for the full dataset. These are estimates. The training cell first downloads the model (about 1 GB).

**Outputs** go to Google Drive, so they survive the session: the model in `models/xlmr/`, plus `reports/step6_xlmr_metrics.json` and `reports/step6_xlmr_predictions.jsonl`. Copy the two report files into the repo's `reports/` folder afterwards.

**How to run:** `Runtime → Run all`. Change the settings in the next cell if needed.

In [ ]:
# --- settings ---------------------------------------------------------------
CODE_SOURCE = "drive"  # "drive": unzip BUNDLE from Drive (code + splits); "github": clone the repo
BUNDLE = "/content/drive/MyDrive/colab_bundle.zip"  # made by `python src/make_colab_bundle.py`

REPO_URL = "https://github.com/cheamenglyzz-spec/khmer-ai-text-detector"
BRANCH = "step2-ai-generation"  # only for CODE_SOURCE = "github"
REPO_DIR = "/content/khmer-ai-text-detector"

DRIVE_ROOT = "/content/drive/MyDrive/khmer-ai-text-detector"
SPLITS_DIR = f"{REPO_DIR}/data/processed/splits" if CODE_SOURCE == "drive" else f"{DRIVE_ROOT}/splits"
OUT_DIR = f"{DRIVE_ROOT}/models/xlmr"
METRICS = f"{DRIVE_ROOT}/reports/step6_xlmr_metrics.json"
PREDICTIONS = f"{DRIVE_ROOT}/reports/step6_xlmr_predictions.jsonl"
CHECKPOINT_DIR = "/content/xlmr_checkpoints"  # local disk: faster than Drive, and nothing temporary is left in Drive

EXTRA_ARGS = ""  # e.g. "--epochs 4 --lr 3e-5", or a quick test: "--limit 64 --max-steps 8 --max-length 128"

## 1. Check the GPU

In [ ]:
!nvidia-smi
import torch

assert torch.cuda.is_available(), "No GPU found: Runtime -> Change runtime type -> T4 GPU, then run again."
print("GPU:", torch.cuda.get_device_name(0))

## 2. Get the code and data

Mounting Drive opens a Google permission window: allow it. Outputs are written to Drive in both modes.

In [ ]:
import os
import zipfile
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

if CODE_SOURCE == "drive":
    assert Path(BUNDLE).exists(), f"{BUNDLE} not found: make it with `python src/make_colab_bundle.py` and drag it into My Drive."
    with zipfile.ZipFile(BUNDLE) as zf:
        zf.extractall(REPO_DIR)
    print(Path(REPO_DIR, "MANIFEST.json").read_text())
else:
    # A public repo needs no token. For a private one set USE_TOKEN = True and:
    #   1. github.com -> Settings -> Developer settings -> Fine-grained tokens: read-only "Contents" access to this repo
    #   2. In Colab, open the key icon (Secrets) on the left, add a secret named GITHUB_TOKEN with the token as value
    #   3. Switch on "Notebook access" for it (it is off for every new notebook)
    USE_TOKEN = False

    clone_url = REPO_URL
    if USE_TOKEN:
        from google.colab import userdata

        try:
            token = userdata.get("GITHUB_TOKEN").strip()
        except userdata.SecretNotFoundError:
            raise RuntimeError("No secret named GITHUB_TOKEN: add it in the Secrets panel (key icon), see the steps above.") from None
        except userdata.NotebookAccessError:
            raise RuntimeError("GITHUB_TOKEN exists but this notebook may not read it: switch on 'Notebook access' in the Secrets panel.") from None
        clone_url = REPO_URL.replace("https://", f"https://x-access-token:{token}@")

    if os.path.isdir(f"{REPO_DIR}/.git"):
        !git -C {REPO_DIR} pull --ff-only
    else:
        !git clone --quiet --depth 1 --branch {BRANCH} {clone_url} {REPO_DIR}
        if os.path.isdir(REPO_DIR):
            !git -C {REPO_DIR} remote set-url origin {REPO_URL}  # don't keep the token in .git/config
    assert os.path.isdir(f"{REPO_DIR}/src"), (
        "Clone failed. Without a token the repo must be public; with one, this usually means it cannot read this repo (fine-grained tokens need the repo selected "
        "and Contents: read) or has expired; also check BRANCH."
    )
%cd {REPO_DIR}

In [ ]:
# Colab already has torch, scikit-learn, pandas and datasets. pip only installs or upgrades what is missing or too old.
!pip install -q "transformers>=4.46" "accelerate>=1.1"

## 3. Check the data

In [ ]:
NEEDED = ["train", "val", "test", "test_ood"]
missing = [f"{n}.jsonl" for n in NEEDED if not (Path(SPLITS_DIR) / f"{n}.jsonl").exists()]
if missing:
    print(f"Missing in {SPLITS_DIR}: {', '.join(missing)}\nCopy them there and run this cell again, or run the upload cell below.")
else:
    for n in NEEDED:
        with open(Path(SPLITS_DIR) / f"{n}.jsonl", encoding="utf-8") as f:
            print(f"{n:<9}{sum(1 for line in f if line.strip()):>6} rows")

**Only if the cell above listed missing files:** this cell asks you to pick `train.jsonl`, `val.jsonl`, `test.jsonl` and `test_ood.jsonl` from your computer (it does nothing when Drive has them). They go to the Colab disk (not Drive), so upload them again in a new session.

In [ ]:
if missing:
    from google.colab import files

    SPLITS_DIR = "/content/splits"
    Path(SPLITS_DIR).mkdir(exist_ok=True)
    for name, data in files.upload().items():
        (Path(SPLITS_DIR) / name).write_bytes(data)
    print(sorted(p.name for p in Path(SPLITS_DIR).glob("*.jsonl")))
else:
    print(f"Using the splits in {SPLITS_DIR}")

## 4. Fine-tune and evaluate

In [ ]:
!python src/finetune.py --splits-dir "{SPLITS_DIR}" --out-dir "{OUT_DIR}" --checkpoint-dir "{CHECKPOINT_DIR}" --metrics "{METRICS}" --predictions "{PREDICTIONS}" {EXTRA_ARGS}

## 5. Results

In [ ]:
import json

import pandas as pd

report = json.loads(Path(METRICS).read_text(encoding="utf-8"))
train = report["training"]
print(f"Device {report['environment']['device']}, trained {train['train_seconds']:.0f}s, best val F1 {train['best_val_f1']} ({train['best_checkpoint']})\n")

print("Metrics (ai = positive class)")
display(pd.DataFrame(report["metrics"]).T[["n", "accuracy", "precision", "recall", "f1", "roc_auc"]])
for split in ("test", "test_ood"):
    print(f"{split}: recall per AI generator, accuracy on human texts per source")
    display(pd.DataFrame(report["breakdowns"][split]["ai_recall_by_generator"]).T)
    display(pd.DataFrame(report["breakdowns"][split]["human_accuracy_by_source"]).T)
print(f"Texts cut off at {report['config']['max_length']} tokens")
display(pd.DataFrame(report["truncation"]).T[["n", "truncated", "fraction", "tokens_median", "tokens_max"]])

## 6. Step 7 evaluation (optional)

Scores the model you just trained with `src/evaluate.py`: metrics per split, breakdowns by generator, source, prompt type and length, the texts cut off at the token limit, the misclassified texts, and two robustness tests (texts cut to their first 300, 600 and 1000 characters; the LLM-paraphrased texts, if `data/eval/paraphrased.jsonl` was in the bundle). It takes a few minutes on a T4. The report goes to Drive next to the step-6 files; copy it into the repo's `reports/` folder afterwards. Set `RUN_STEP7 = False` to skip this section.

In [ ]:
RUN_STEP7 = True  # False skips this section
EVAL_REPORT = f"{DRIVE_ROOT}/reports/step7_eval_xlmr.json"

if RUN_STEP7:
    import json

    MAX_LEN = json.loads(Path(METRICS).read_text(encoding="utf-8"))["config"]["max_length"]  # same token limit as in training
    !python src/evaluate.py --model xlmr --model-path "{OUT_DIR}" --splits-dir "{SPLITS_DIR}" --max-length {MAX_LEN} --out "{EVAL_REPORT}"